# cr-geo-models: Colab launcher

A thin launcher. All modelling code lives in the `ml/` package at a pinned commit. Run the cells in order.
Before you start: note the compute-unit balance in `docs/colab_ledger.md`. When done: **Runtime → Disconnect and delete runtime** (an idle runtime still uses units).

In [ ]:
# 1. Check the runtime
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
GPU_TYPE = gpu.stdout.strip() if gpu.returncode == 0 and gpu.stdout.strip() else "cpu"
print("GPU:", GPU_TYPE)

In [ ]:
# 2. Get the code at a pinned commit (fill in before running; never a branch name)
REPO_URL = ""     # the public GitHub URL of cr-geo-models
COMMIT_SHA = ""   # full 40-character commit SHA
assert REPO_URL.startswith("https://"), "Set REPO_URL"
assert len(COMMIT_SHA) == 40, "Set COMMIT_SHA to a full commit SHA"
!git clone --quiet "$REPO_URL" /content/cr-geo-models
%cd /content/cr-geo-models
!git checkout --quiet "$COMMIT_SHA" && git rev-parse HEAD

In [ ]:
# 3. Install pinned dependencies
!pip install --quiet -r requirements-dev.txt
# Training gates add requirements-ml.txt (pinned from a resolved Colab install):
# !pip install --quiet -r requirements-ml.txt

In [ ]:
# 4. Secrets: read from Colab Secrets (key icon on the left), never typed here. Fails closed if missing.
from ml.tokens import require_secret
# Training needs no write token. Only the separate publishing step does, e.g.:
# HF_TOKEN = require_secret("HF_TOKEN")

In [ ]:
# 5. Mount storage (staged data, checkpoints and the MLflow file store live here)
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/cr-geo-models"  # owner confirms the folder

In [ ]:
# 6. Check the run record and the staged data before spending GPU time
CONFIG = ""  # e.g. configs/<run>.yaml, added by the training gate
assert CONFIG, "Set CONFIG"
from ml.config import load_config
from ml.manifest import load_manifest, verify
from ml.run_metadata import build_run_metadata
record = build_run_metadata(".", CONFIG, gpu_type=GPU_TYPE)
verify(load_manifest(load_config(CONFIG)["data_manifest"]), DATA_ROOT)
record

In [ ]:
# 7. Train and evaluate: the training gate adds this command, e.g.
# !python -m ml.train --config "$CONFIG" --data-root "$DATA_ROOT"
# !python -m ml.evaluate --config "$CONFIG" --data-root "$DATA_ROOT"
print("No training command yet: added in the training gate.")

# 8. Finish
Record the units used in `docs/colab_ledger.md`, then **Runtime → Disconnect and delete runtime**.
Clear all outputs before committing any change to this notebook.